<a href="https://colab.research.google.com/github/amitpoa/Spam-Not-Spam-Classification-RNN-/blob/main/Interactive_Inferance_(spam_rnn_classifier).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive Inferance

In [6]:
# =====================================================================
#  Interactive Web UI Test Application (Gradio Interface)
# =====================================================================
# ── Step 1: Force Install Gradio and Text Processing Utilities ───────
!pip install -q gradio contractions emoji beautifulsoup4 nltk scikit-learn

import os
import re
import string
import pickle
import urllib.request
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

import contractions
import emoji
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Setup NLTK dependencies
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
STOP_WORDS = set(stopwords.words("english"))

# ── FIXED RAW PUBLIC GITHUB DIRECT DOWNLOAD URLS ─────────────────────
MODEL_URL = "https://github.com/amitpoa/Spam-Not-Spam-Classification-RNN-/releases/download/v1.0/simplernn_model_1.keras"
TOKENIZER_URL = "https://github.com/amitpoa/Spam-Not-Spam-Classification-RNN-/releases/download/v1.0/tokenizer.pickle"

# Local workspace paths on Colab's hard drive
MODEL_PATH = "simplernn_model_1.keras"
TOKENIZER_PATH = "tokenizer.pickle"
MAX_LEN = 100

# ── Step 2: Automated Public Asset Downloader ────────────────────────
def download_assets():
    if not os.path.exists(MODEL_PATH):
        print("📥 Downloading production model weights from GitHub Release...")
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    if not os.path.exists(TOKENIZER_PATH):
        print("📥 Downloading text vocabulary tokenizer dictionary...")
        urllib.request.urlretrieve(TOKENIZER_URL, TOKENIZER_PATH)
    print("✅ All assets present locally in workspace.")

# Execute download before loading anything
download_assets()

# ── Step 3: Preprocessing Engine ─────────────────────────────────────
def preprocess_text(text: str) -> str:
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = text.encode("ascii", errors="ignore").decode("ascii")
    text = contractions.fix(text)
    text = text.lower()
    text = emoji.demojize(text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b\d+\b", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in STOP_WORDS]
    return " ".join(tokens)

# ── Step 4: Load Local Assets into Global State ──────────────────────
print("💾 Initializing Saved Model & Tokenizer locally...")
if os.path.exists(MODEL_PATH) and os.path.exists(TOKENIZER_PATH):
    UI_MODEL = load_model(MODEL_PATH)
    with open(TOKENIZER_PATH, "rb") as f:
        UI_TOKENIZER = pickle.load(f)
    print("✅ Model assets successfully loaded! Launching UI interface...")
else:
    print("❌ Error: Missing saved components. Check paths before launching Gradio.")
    UI_MODEL, UI_TOKENIZER = None, None

# ── Step 5: Core Prediction UI Wrapper Function ──────────────────────
def gradio_predict_engine(raw_message):
    """
    Interface function that maps a text input string to formatted
    dictionary probabilities for the Gradio label dashboard components.
    """
    if UI_MODEL is None or UI_TOKENIZER is None:
        return {"Error: Model assets missing from drive": 1.0}

    if not raw_message.strip():
        return {"Please enter text context...": 1.0}

    # Execute backend preprocessing pipeline
    clean = preprocess_text(raw_message)
    seq = UI_TOKENIZER.texts_to_sequences([clean])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")

    # Run forward pass prediction
    score = float(UI_MODEL.predict(padded, verbose=0)[0][0])

    # Map raw continuous probability scale back into label balances
    # score -> 1.0 means high confidence Ham, score -> 0.0 means high confidence Spam
    ham_prob = score
    spam_prob = 1.0 - score

    return {
        "Ham (Legitimate Message)": ham_prob,
        "Spam (Suspicious Content)": spam_prob
    }

# ── Step 6: Construct Interactive Layout ─────────────────────────────
import gradio as gr

# Setup preset quick-test scenarios for the examiner
example_scenarios = [
    ["CONGRATULATIONS! Your mobile number was selected as the lucky winner of a $5,000 Walmart Gift Card! Text 'WIN' to 88300 immediately to claim your free reward now!"],
    ["Hey! Are you free to grab some coffee after work today? Let me know if that time works for you, otherwise we can reschedule for tomorrow morning."],
    ["URGENT SECURITY ALERT: Your online banking access has been temporarily locked due to suspicious login attempts. Click here: http://secure-bank-verify.com to unlock your account immediately."]
]

app_interface = gr.Interface(
    fn=gradio_predict_engine,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Type or paste your SMS/Email message contents here...",
        label="Input Communication Text Content"
    ),
    outputs=gr.Label(num_top_classes=2, label="Classification Analysis Output"),
    examples=example_scenarios,
    title="🛡️ Stacked Spam Detection System",
    description="An interactive playground running your production-trained Stacked Recurrent Neural Network model. Type custom messages below or select an example to evaluate standard context classification.",
    theme="soft"
)

# Launch interface inline directly inside your notebook output block
# Change share=True if you want to generate a public link to give to your examiner
app_interface.launch(inline=True, share=False)

✅ All assets present locally in workspace.
💾 Initializing Saved Model & Tokenizer locally...
✅ Model assets successfully loaded! Launching UI interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>